In [2]:
import pandas as pd

usecols = [
    'is_fraud', 'source',
    'spending_deviation_score', 'velocity_score', 'geo_anomaly_score', 'amount_ngn',
    'geospatial_velocity_anomaly', 'txn_hour', 'is_weekend', 'is_salary_week', 'is_night_txn',
    'device_seen_count', 'is_device_shared', 'ip_seen_count', 'is_ip_shared',
    'user_txn_count_total', 'user_avg_txn_amt', 'user_std_txn_amt', 'user_txn_frequency_24h',
    'txn_count_last_1h', 'txn_count_last_24h', 'total_amount_last_1h',
    'time_since_last', 'avg_gap_between_txns',
    'merchant_fraud_rate', 'channel_risk_score', 'persona_fraud_risk', 'location_fraud_risk',
    'is_high_risk_state', 'is_high_risk_lga', 'is_ato_risk',
]
dtype_map = {
    'is_fraud': 'bool', 'source': 'category',
    'amount_ngn': 'float32', 'spending_deviation_score': 'float32',
    'velocity_score': 'int32', 'geo_anomaly_score': 'float32',
    'txn_hour': 'int8', 'is_weekend': 'int8', 'is_salary_week': 'int8', 'is_night_txn': 'int8',
    'device_seen_count': 'int32', 'is_device_shared': 'int8', 'ip_seen_count': 'int32', 'is_ip_shared': 'int8',
    'user_txn_count_total': 'int32', 'user_avg_txn_amt': 'float32', 'user_std_txn_amt': 'float32',
    'user_txn_frequency_24h': 'int32', 'txn_count_last_1h': 'int32', 'txn_count_last_24h': 'int32',
    'total_amount_last_1h': 'float32', 'time_since_last': 'float32', 'avg_gap_between_txns': 'float32',
    'merchant_fraud_rate': 'float32', 'channel_risk_score': 'float32', 'persona_fraud_risk': 'float32',
    'location_fraud_risk': 'float32', 'is_high_risk_state': 'int8', 'is_high_risk_lga': 'int8',
    'is_ato_risk': 'int8', 'geospatial_velocity_anomaly': 'bool',
}
df = pd.read_csv('../data/processed/nigerian_transactions_clean.csv', usecols=usecols, dtype=dtype_map)
print(df.shape)

(5086000, 31)


In [3]:
from sklearn.model_selection import train_test_split
import gc

df_sample, _ = train_test_split(
    df, train_size=1_000_000, random_state=42, stratify=df['is_fraud']
)
del df
gc.collect()

print("Sampled shape:", df_sample.shape, " Fraud rate:", df_sample['is_fraud'].mean())

Sampled shape: (1000000, 31)  Fraud rate: 0.035992


In [4]:
FEATURES = [
    'spending_deviation_score', 'velocity_score', 'geo_anomaly_score', 'amount_ngn',
    'geospatial_velocity_anomaly', 'txn_hour', 'is_weekend', 'is_salary_week', 'is_night_txn',
    'device_seen_count', 'is_device_shared', 'ip_seen_count', 'is_ip_shared',
    'user_txn_count_total', 'user_avg_txn_amt', 'user_std_txn_amt', 'user_txn_frequency_24h',
    'txn_count_last_1h', 'txn_count_last_24h', 'total_amount_last_1h',
    'time_since_last', 'avg_gap_between_txns',
    'merchant_fraud_rate', 'channel_risk_score', 'persona_fraud_risk', 'location_fraud_risk',
    'is_high_risk_state', 'is_high_risk_lga', 'is_ato_risk',
]

X = df_sample[FEATURES]
y = df_sample['is_fraud']
source = df_sample['source']

X_train, X_test, y_train, y_test, source_train, source_test = train_test_split(
    X, y, source, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape, " Fraud rate:", y_train.mean())
print("Test shape:", X_test.shape, " Fraud rate:", y_test.mean())
print(source_test.value_counts())

Train shape: (800000, 29)  Fraud rate: 0.0359925
Test shape: (200000, 29)  Fraud rate: 0.03599
source
real         196592
synthetic      3408
Name: count, dtype: int64


In [5]:
from sklearn.ensemble import RandomForestClassifier
import time

model = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=2,
    verbose=1
)

start = time.time()
model.fit(X_train, y_train)
print(f"\nTraining took {time.time() - start:.1f} seconds")

[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:   58.8s



Training took 64.7 seconds


[Parallel(n_jobs=2)]: Done  50 out of  50 | elapsed:  1.1min finished


In [6]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test)

print("=== Overall performance (real + synthetic combined) ===")
print(classification_report(y_test, y_pred, target_names=['Not Fraud', 'Fraud']))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

print("\n=== Performance on REAL data only ===")
real_mask = (source_test == 'real').values
print(classification_report(y_test[real_mask], y_pred[real_mask], target_names=['Not Fraud', 'Fraud']))

print("\n=== Performance on SYNTHETIC data only ===")
synth_mask = (source_test == 'synthetic').values
print(classification_report(y_test[synth_mask], y_pred[synth_mask], target_names=['Not Fraud', 'Fraud']))

[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.4s
[Parallel(n_jobs=2)]: Done  50 out of  50 | elapsed:    0.5s finished


=== Overall performance (real + synthetic combined) ===
              precision    recall  f1-score   support

   Not Fraud       0.99      0.21      0.35    192802
       Fraud       0.04      0.97      0.08      7198

    accuracy                           0.24    200000
   macro avg       0.52      0.59      0.22    200000
weighted avg       0.96      0.24      0.34    200000

Confusion matrix:
 [[ 40943 151859]
 [   237   6961]]

=== Performance on REAL data only ===
              precision    recall  f1-score   support

   Not Fraud       0.99      0.21      0.35    189547
       Fraud       0.04      0.97      0.08      7045

    accuracy                           0.24    196592
   macro avg       0.52      0.59      0.22    196592
weighted avg       0.96      0.24      0.34    196592


=== Performance on SYNTHETIC data only ===
              precision    recall  f1-score   support

   Not Fraud       0.99      0.25      0.40      3255
       Fraud       0.05      0.93      0.10 

In [7]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,           # deeper - let it learn finer splits within "fraud-like" groups
    min_samples_leaf=5,     # avoid overfitting to single noisy rows despite the depth increase
    class_weight='balanced_subsample',  # rebalances per-tree bootstrap sample, less overcorrection than global 'balanced'
    random_state=42,
    n_jobs=2,
    verbose=1
)

start = time.time()
model.fit(X_train, y_train)
print(f"\nTraining took {time.time() - start:.1f} seconds")

y_pred = model.predict(X_test)
print("\n=== Overall performance ===")
print(classification_report(y_test, y_pred, target_names=['Not Fraud', 'Fraud']))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:  1.9min
[Parallel(n_jobs=2)]: Done 100 out of 100 | elapsed:  4.4min finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.



Training took 267.2 seconds


[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.8s
[Parallel(n_jobs=2)]: Done 100 out of 100 | elapsed:    1.6s finished



=== Overall performance ===
              precision    recall  f1-score   support

   Not Fraud       0.97      0.80      0.87    192802
       Fraud       0.04      0.24      0.07      7198

    accuracy                           0.78    200000
   macro avg       0.50      0.52      0.47    200000
weighted avg       0.93      0.78      0.84    200000

Confusion matrix:
 [[153768  39034]
 [  5449   1749]]


In [8]:
print("Fraud rate by is_ato_risk (train set):")
print(y_train.groupby(X_train['is_ato_risk']).mean())

print("\nFeature importances from the trained model:")
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print(importances)

Fraud rate by is_ato_risk (train set):
is_ato_risk
0    0.000000
1    0.043844
Name: is_fraud, dtype: float64

Feature importances from the trained model:
is_ato_risk                    0.365510
merchant_fraud_rate            0.050292
location_fraud_risk            0.050096
amount_ngn                     0.047805
spending_deviation_score       0.047592
user_avg_txn_amt               0.044920
total_amount_last_1h           0.044782
geo_anomaly_score              0.041483
ip_seen_count                  0.040443
time_since_last                0.040164
avg_gap_between_txns           0.039438
user_std_txn_amt               0.037906
txn_hour                       0.031014
velocity_score                 0.029325
user_txn_count_total           0.023634
channel_risk_score             0.012589
device_seen_count              0.007879
txn_count_last_24h             0.007160
user_txn_frequency_24h         0.007159
txn_count_last_1h              0.006993
persona_fraud_risk             0.006575
is_we

In [9]:
from sklearn.metrics import precision_recall_curve, precision_score, recall_score
import numpy as np

y_proba = model.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)

f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_idx = np.argmax(f1_scores)
print(f"Best F1 threshold: {thresholds[best_idx]:.4f}")
print(f"At that threshold — Precision: {precisions[best_idx]:.4f}, Recall: {recalls[best_idx]:.4f}")

for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    preds_at_t = (y_proba >= t).astype(int)
    p = precision_score(y_test, preds_at_t, zero_division=0)
    r = recall_score(y_test, preds_at_t, zero_division=0)
    print(f"Threshold {t}: precision={p:.3f}, recall={r:.3f}")

[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    1.0s
[Parallel(n_jobs=2)]: Done 100 out of 100 | elapsed:    2.4s finished


Best F1 threshold: 0.2982
At that threshold — Precision: 0.0440, Recall: 0.9764
Threshold 0.1: precision=0.044, recall=1.000
Threshold 0.2: precision=0.044, recall=0.998
Threshold 0.3: precision=0.044, recall=0.975
Threshold 0.4: precision=0.044, recall=0.809
Threshold 0.5: precision=0.043, recall=0.243
Threshold 0.6: precision=1.000, recall=0.001
Threshold 0.7: precision=0.000, recall=0.000


In [10]:
import joblib
joblib.dump(model, '../models/fraud_model_v1.pkl')
print("Model saved.")

Model saved.
